In [1]:
import pandas as pd
from itertools import islice
from pyexpat import features
import torch
import numpy as np
from torchvision import transforms
import pandas as pd
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import Parallel, delayed
from scipy.stats import skew
from sklearn.preprocessing import MinMaxScaler


In [2]:
train_df = pd.read_csv('./dataset/train.csv')
det_df = pd.read_csv('./dataset/train_det.csv')
gt_df = pd.read_csv('./dataset/train_gt.csv')

In [3]:
test_df = pd.read_csv('./dataset/test.csv')
test_det_df = pd.read_csv('./dataset/test_det.csv')

In [4]:
train_df.head()

,Unnamed: 0,video_id,video_name,image_width,image_height,frame_id,image_path
0,0,0,ADL-Rundle-6,1920,1080,0,./dataset/train/ADL-Rundle-6/img1/000001.jpg
1,1,0,ADL-Rundle-6,1920,1080,1,./dataset/train/ADL-Rundle-6/img1/000002.jpg
2,2,0,ADL-Rundle-6,1920,1080,2,./dataset/train/ADL-Rundle-6/img1/000003.jpg
3,3,0,ADL-Rundle-6,1920,1080,3,./dataset/train/ADL-Rundle-6/img1/000004.jpg
4,4,0,ADL-Rundle-6,1920,1080,4,./dataset/train/ADL-Rundle-6/img1/000005.jpg


In [5]:
det_df.head()

,Unnamed: 0,frame_id,object_id,x_coordinate,y_coordinate,width,height,confidence,x_init,y_init,z_init,video_id
0,0,0,-1,1689,385,146.620,332.710,67.567,-1.0,-1.0,-1.0,0
1,1,0,-1,1303,503,61.514,139.590,29.439,-1.0,-1.0,-1.0,0
2,2,0,-1,1258,569,40.123,91.049,19.601,-1.0,-1.0,-1.0,0
3,3,0,-1,31,525,113.370,257.270,17.013,-1.0,-1.0,-1.0,0
4,4,0,-1,1800,483,94.660,214.810,11.949,-1.0,-1.0,-1.0,0


In [ ]:
import torchvision.transforms.functional as F
import random

class DataAugmentation:
    def __init__(self, is_train=True):
        self.is_train = is_train

    def __call__(self, image, target):
        if not self.is_train:
            return image, target

        # 1. Color Jittering (Only affects image)
        if random.random() > 0.5:
            brightness = random.uniform(0.8, 1.2)
            contrast = random.uniform(0.8, 1.2)
            image = F.adjust_brightness(image, brightness)
            image = F.adjust_contrast(image, contrast)

        # 2. Horizontal Flip (Affects image and boxes)
        if random.random() > 0.5:
            image = F.hflip(image)
            width = image.shape[-1]
            boxes = target["boxes"]
            # Flip: x_min_new = width - x_max_old; x_max_new = width - x_min_old
            boxes[:, [0, 2]] = width - boxes[:, [2, 0]]
            target["boxes"] = boxes

        # 3. Random Crop (Simplified as "Random Resized Crop" style)
        # Note: True random cropping requires complex box filtering. 
        # For object detection, it's safer to use ColorJitter and Flips first.
        
        return image, target

In [6]:
import torch
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import DataLoader, Dataset
import pandas as pd
import numpy as np
import cv2
import os
from PIL import Image

# --- Configuration ---
# DEVICE = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
DEVICE = 'cpu'
NUM_EPOCHS = 10
BATCH_SIZE = 4  # Start with 4 for 1080p images on 16GB VRAM; increase to 8 if stable
LEARNING_RATE = 0.005
NUM_CLASSES = 2  # 1 class (object) + 1 background. Change if you have specific classes.
# If your dataset distinguishes between cars/pedestrians, increase this.

# --- 1. Custom Dataset Class ---
class CustomMOTDataset(Dataset):
    def __init__(self, root_dir, index_csv, gt_csv, transforms=None):
        """
        Args:
            root_dir (string): Directory with all the images.
            index_csv (string): Path to train.csv (frames list).
            gt_csv (string): Path to train_gt.csv (annotations).
            transforms (callable, optional): Optional transform to be applied on a sample.
        """
        self.root_dir = root_dir
        self.transforms = transforms

        # Load DataFrames
        self.frames_df = pd.read_csv(index_csv)
        self.gt_df = pd.read_csv(gt_csv)

        # Ensure we can map frames to annotations quickly
        # Assuming both CSVs share 'video_id' and 'frame_id' or similar keys
        # We create a dictionary for faster lookup during training
        self.image_data = self.frames_df.reset_index(drop=True)

    def __getitem__(self, idx):
        # 1. Get Image Info
        row = self.image_data.iloc[idx]
        video_id = row['video_id']
        frame_id = row['frame_id']

        img_path = row['image_path']

        if not os.path.exists(img_path):
            raise FileNotFoundError(f"Image not found: {img_path}")

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB).astype(np.float32)
        image /= 255.0  # Normalize to [0, 1]

        # 2. Get Annotations for this frame
        # Filter GT dataframe for this specific video and frame
        # Adjust column names 'video_id'/'frame_id' to match your CSV headers exactly
        boxes_data = self.gt_df[
            (self.gt_df['video_id'] == video_id) &
            (self.gt_df['frame_id'] == frame_id)
        ]

        boxes = []
        labels = []

        if len(boxes_data) > 0:
            for _, box_row in boxes_data.iterrows():
                x_min = float(box_row['x_coordinate'])
                y_min = float(box_row['y_coordinate'])
                w = float(box_row['width'])
                h = float(box_row['height'])

                x_max = x_min + w
                y_max = y_min + h

                boxes.append([x_min, y_min, x_max, y_max])
                labels.append(1) # Assuming all objects are class 1. Change if needed.
        else:
            # Handle negative samples (images with no objects)
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)

        # Convert to Tensors
        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)

        image_id = torch.tensor([idx])
        area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
        iscrowd = torch.zeros((len(labels),), dtype=torch.int64)

        target = {}
        target["boxes"] = boxes
        target["labels"] = labels
        target["image_id"] = image_id
        target["area"] = area
        target["iscrowd"] = iscrowd

        # Convert image to tensor (C, H, W)
        image = torch.as_tensor(image.transpose((2, 0, 1)), dtype=torch.float32)

        return image, target

    def __len__(self):
        return len(self.image_data)

# --- 2. Model Definition ---
def get_model(num_classes):
    # Load a model pre-trained on COCO
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")

    # Replace the classifier with a new one, that has user-defined num_classes
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    return model

# --- 3. Training Loop ---
def train_one_epoch(model, optimizer, data_loader, device, epoch):
    model.train()
    total_loss = 0

    for i, (images, targets) in enumerate(data_loader):
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # Forward pass (returns losses in train mode)
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        # Backward pass
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        total_loss += losses.item()

        if i % 10 == 0:
            print(f"Epoch: {epoch}, Iter: {i}, Loss: {losses.item():.4f}")

    print(f"Epoch {epoch} finished. Average Loss: {total_loss / len(data_loader):.4f}")

# --- 4. Main Execution ---
def collate_fn(batch):
    return tuple(zip(*batch))


In [ ]:
DATA_DIR = './dataset'
TRAIN_CSV = './dataset/train.csv'
GT_CSV = './dataset/train_gt.csv'

train_transform = DataAugmentation(is_train=True)
print(f"Initializing Dataset on {DEVICE}...")
dataset = CustomMOTDataset(DATA_DIR, TRAIN_CSV, GT_CSV, transforms=train_transform)

# DataLoader
data_loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=4,
        collate_fn=collate_fn
    )

print("Initializing Model...")
model = get_model(NUM_CLASSES)
model.to(DEVICE)

# Optimizer
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=LEARNING_RATE, momentum=0.9, weight_decay=0.0005)

# Learning Rate Scheduler
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

print("Starting Training...")
for epoch in range(NUM_EPOCHS):
    train_one_epoch(model, optimizer, data_loader, DEVICE, epoch)
    lr_scheduler.step()

    torch.save(model.state_dict(), f"faster_rcnn_epoch_{epoch}.pth")

print("Training Complete. Model saved.")

Initializing Dataset on cpu...
Initializing Model...
Starting Training...
Epoch: 0, Iter: 0, Loss: 1.7186
Epoch: 0, Iter: 10, Loss: 0.8463
